In [ ]:
# --- Importaciones ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
from xgboost import XGBRegressor
import scipy.stats as stats

import sklearn
print(f'sklearn:  {sklearn.__version__}')
print(f'xgboost:  {xgb.__version__}')
print(f'numpy:    {np.__version__}')

In [ ]:
# --- California Housing: train/val/test split ------------------------
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

semilla = 42

# Split 1: separar el 30% para val+test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=semilla
)
# Split 2: dividir ese 30% en mitades iguales -> val=15%, test=15%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=semilla
)

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')


In [ ]:
# --- GradientBoostingRegressor (sklearn) ----------------------------
t0 = time.time()
gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=semilla,
)
gb.fit(X_train, y_train)
t_gb = time.time() - t0

# Fix: remove 'squared=False' and take the square root for RMSE
rmse_gb = np.sqrt(mean_squared_error(y_test, gb.predict(X_test)))
r2_gb   = r2_score(y_test, gb.predict(X_test))
print(f'GB sklearn | RMSE: {rmse_gb:.4f} R²: {r2_gb:.4f} t: {t_gb:.1f}s')

In [ ]:
# --- XGBRegressor básico --------------------------------------------
t0 = time.time()
xgb_base = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=semilla,
    n_jobs=-1,
    verbosity=0,      # silencia los logs de entrenamiento
)
xgb_base.fit(X_train, y_train)
t_xgb = time.time() - t0

rmse_xgb = np.sqrt(mean_squared_error(
    y_test, xgb_base.predict(X_test)
))
r2_xgb = r2_score(y_test, xgb_base.predict(X_test))
print(f'XGBoost    | RMSE: {rmse_xgb:.4f} R²: {r2_xgb:.4f} t: {t_xgb:.1f}s')


In [ ]:
# --- XGBoost con early stopping -------------------------------------
xgb_es = XGBRegressor(
    n_estimators=1000,      # máximo posible; early stopping lo recorta
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=semilla,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=30,  # para si no mejora en 30 rondas
    eval_metric='rmse',
)

xgb_es.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f'Mejor n_estimators: {xgb_es.best_iteration}')
rmse_es = np.sqrt(mean_squared_error(
    y_test, xgb_es.predict(X_test)
))
r2_es = r2_score(y_test, xgb_es.predict(X_test))
print(f'XGBoost+ES | RMSE: {rmse_es:.4f} R²: {r2_es:.4f}')


In [ ]:
# --- Curva de pérdida train vs eval ---------------------------------
# Necesitamos re-entrenar capturando el historial de pérdidas
xgb_hist = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    random_state=semilla,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=30,
    eval_metric='rmse',
)
xgb_hist.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
)

historial = xgb_hist.evals_result()
rmse_train = historial['validation_0']['rmse']
rmse_val   = historial['validation_1']['rmse']

plt.figure(figsize=(9, 4))
plt.plot(rmse_train, label='Train', alpha=0.8)
plt.plot(rmse_val,   label='Validación', alpha=0.8)
plt.axvline(xgb_hist.best_iteration, color='red',
            linestyle='--', label='Early stop')
plt.xlabel('Número de árboles')
plt.ylabel('RMSE')
plt.title('Curva de pérdida — XGBoost')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Importancia de features ----------------------------------------
importancias = pd.Series(
    xgb_es.feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)

importancias.plot(
    kind='barh', figsize=(8, 5), color='steelblue'
)
plt.xlabel('Importancia (gain)')
plt.title('Feature importance — XGBoost California Housing')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(importancias.round(4))

In [ ]:
# --- RandomizedSearchCV sobre XGBoost -------------------------------
# Combinamos train + val para la búsqueda (CV interno)
X_search = pd.concat([X_train, X_val])
y_search = pd.concat([y_train, y_val])

# Mezclar listas discretas con distribuciones continuas es habitual:
# n_estimators se prueba en valores concretos (200/400/600)
# learning_rate se explora en rango continuo con loguniform
param_dist = {
    'n_estimators':     [200, 400, 600],
    'learning_rate':    stats.loguniform(0.01, 0.3),
    'max_depth':        [3, 4, 5, 6],
    'subsample':        stats.uniform(0.6, 0.4),
    'colsample_bytree': stats.uniform(0.6, 0.4),
    'reg_alpha':        stats.loguniform(0.001, 1.0),
    'reg_lambda':       stats.loguniform(0.1, 10.0),
}

rs = RandomizedSearchCV(
    XGBRegressor(
        random_state=semilla, n_jobs=-1, verbosity=0
    ),
    param_distributions=param_dist,
    n_iter=25,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=semilla,
)
rs.fit(X_search, y_search)

print(f'Mejores params: {rs.best_params_}')
rmse_tuned = np.sqrt(mean_squared_error(
    y_test, rs.best_estimator_.predict(X_test)
))
print(f'RMSE en test (tuned): {rmse_tuned:.4f}')

In [ ]:
# --- Entrenar todos los modelos para comparar -----------------------
modelos = {
    'Lineal': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, random_state=semilla, n_jobs=-1
    ),
    'GB sklearn': GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1,
        max_depth=4, random_state=semilla,
    ),
    'XGBoost+ES': xgb_es,   # ya entrenado
}

print(f'{'Modelo':<18} {'RMSE':>8} {'R²':>8} {'Fit(s)':>8}')
for nombre, modelo in modelos.items():
    if nombre != 'XGBoost+ES':
        t0 = time.time()
        modelo.fit(X_train, y_train)
        t = time.time() - t0
    else:
        t = t_xgb
    pred = modelo.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)
    print(f'{nombre:<18} {rmse:>8.4f} {r2:>8.4f} {t:>8.1f}')

In [ ]:
# --- Pipeline sklearn + XGBoost -------------------------------------
# XGBoost no requiere escalado, pero el pipeline permite
# añadir pasos de preprocesamiento si el dataset lo requiere.
pipeline_xgb = Pipeline([
    ('modelo', XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=semilla,
        n_jobs=-1,
        verbosity=0,
    )),
])

pipeline_xgb.fit(X_train, y_train)
rmse_pipe = np.sqrt(mean_squared_error(
    y_test, pipeline_xgb.predict(X_test)
))
print(f'Pipeline XGBoost | RMSE: {rmse_pipe:.4f}')